# 01 · Base pipeline — where does Patna's water pool?

**What this notebook does:** downloads a bare-earth DEM for Patna, runs depression (sink)
detection + flow accumulation, ranks candidate water-pooling sites by volume, delineates the
catchment draining into each one, and renders everything on an interactive map.

**Outputs (used by notebooks 02–05):** `dem.tif`, `depth.tif`, `acc.tif`, `worldcover.tif`,
`catchment_labels.tif`, `sinks.csv`, `sinks.geojson`.

**Before running:**
1. Sign up for Google Earth Engine (free, non-commercial): https://earthengine.google.com
2. Create/choose a Google Cloud project and register it for Earth Engine, then put its ID in
   `PROJECT_ID` below.
3. (Optional) Mount Google Drive in the next cell so outputs persist between sessions.

In [ ]:
import sys, subprocess, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("pysheds", "rasterio", "geopandas")

# Optional: persist outputs to Google Drive. Uncomment the two lines below.
# from google.colab import drive
# drive.mount('/content/drive')

WORK = "/content/drive/MyDrive/floodtwin" if os.path.isdir("/content/drive/MyDrive") else "/content/floodtwin"
os.makedirs(WORK, exist_ok=True)
print("Working directory:", WORK)

In [ ]:
import ee

PROJECT_ID = "your-cloud-project-id"   # <-- EDIT: your Earth Engine cloud project ID

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# Area of interest: greater Patna (~28 km x 15 km). Edit for another city.
AOI = (85.02, 25.54, 85.30, 25.68)          # lon_min, lat_min, lon_max, lat_max
region = ee.Geometry.Rectangle(list(AOI))
SCALE = 30                                   # metres per pixel
CELL_AREA = SCALE * SCALE                    # m^2, nominal

In [ ]:
import requests

def download_ee_image(img, path, scale=SCALE, bands=None):
    """Download a (small) EE image as GeoTIFF via getDownloadURL."""
    if bands is not None:
        img = img.select(bands)
    url = img.getDownloadURL({
        "region": region, "scale": scale, "format": "GEO_TIFF", "crs": "EPSG:4326",
    })
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    print("saved", path, f"({len(r.content)/1e6:.1f} MB)")

# DEM: FABDEM (buildings/forests removed — best free option for urban floodplains),
# falling back to Copernicus GLO-30 if the community asset is unavailable.
try:
    dem_img = ee.ImageCollection("projects/sat-io/open-datasets/FABDEM").filterBounds(region).mosaic()
    download_ee_image(dem_img, f"{WORK}/dem.tif")
    print("DEM source: FABDEM")
except Exception as e:
    print("FABDEM failed (", e, ") — falling back to Copernicus GLO-30")
    dem_img = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").filterBounds(region).mosaic()
    download_ee_image(dem_img, f"{WORK}/dem.tif")

# Land cover (ESA WorldCover 2021, 10 m) and permanent water (JRC), both resampled to 30 m.
download_ee_image(ee.ImageCollection("ESA/WorldCover/v200").first(), f"{WORK}/worldcover.tif")
download_ee_image(ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0),
                  f"{WORK}/jrc_occurrence.tif")

## Hydrological conditioning
Fill pits and depressions, resolve flats, compute D8 flow direction and flow accumulation.
The *depression depth* (filled DEM minus raw DEM) tells us where water can physically pond
and how deep.

In [ ]:
import numpy as np
from pysheds.grid import Grid

grid = Grid.from_raster(f"{WORK}/dem.tif")
dem = grid.read_raster(f"{WORK}/dem.tif")

pit_filled = grid.fill_pits(dem)
flooded = grid.fill_depressions(pit_filled)
depth = np.asarray(flooded, dtype=np.float64) - np.asarray(dem, dtype=np.float64)

inflated = grid.resolve_flats(flooded)
fdir = grid.flowdir(inflated)
acc = grid.accumulation(fdir)

print("max ponding depth (m):", float(np.nanmax(depth)))
print("cells with >0.15 m ponding:", int((depth > 0.15).sum()))

## Label sinks and rank them
Connected ponding regions deeper than 15 cm become candidate sites. We drop anything that is
already permanent water (the Ganga, ponds) using the JRC occurrence layer, then rank by
storable volume.

In [ ]:
import rasterio
from rasterio.transform import xy as px2coord
from scipy import ndimage
import pandas as pd

with rasterio.open(f"{WORK}/jrc_occurrence.tif") as src:
    jrc = src.read(1)
with rasterio.open(f"{WORK}/dem.tif") as src:
    transform, shape = src.transform, src.shape

# Align shapes defensively (downloads can differ by a pixel).
r = min(depth.shape[0], jrc.shape[0]); c = min(depth.shape[1], jrc.shape[1])
depth_a, jrc_a = depth[:r, :c], jrc[:r, :c]

MIN_DEPTH, MIN_CELLS = 0.15, 10
mask = (depth_a > MIN_DEPTH) & (jrc_a < 50)          # ponding, not permanent water
labels, n = ndimage.label(mask)
print("raw sink candidates:", n)

rows = []
for sid in range(1, n + 1):
    cells = labels == sid
    ncells = int(cells.sum())
    if ncells < MIN_CELLS:
        continue
    vol = float(depth_a[cells].sum() * CELL_AREA)     # m^3 of storable water
    dmax = float(depth_a[cells].max())
    # deepest cell = natural pour point of the depression
    rr, cc = np.unravel_index(np.argmax(np.where(cells, depth_a, -1)), depth_a.shape)
    lon, lat = px2coord(transform, rr, cc)
    rows.append(dict(sink_id=sid, lat=lat, lon=lon, row=int(rr), col=int(cc),
                     area_m2=ncells * CELL_AREA, max_depth_m=round(dmax, 2),
                     volume_m3=round(vol, 0)))

sinks = pd.DataFrame(rows).sort_values("volume_m3", ascending=False).reset_index(drop=True)
TOP_N = 25
top = sinks.head(TOP_N).copy()
print(top[["sink_id", "lat", "lon", "max_depth_m", "area_m2", "volume_m3"]].to_string())

## Catchments — how much land drains into each sink
The catchment area multiplied by runoff depth (notebook 04) gives the inflow volume each sink
must handle. We also save a label raster so later notebooks can aggregate rainfall per catchment.

In [ ]:
catch_labels = np.zeros(depth_a.shape, dtype=np.int32)
catch_areas = {}
for _, s in top.iterrows():
    lon, lat = s.lon, s.lat
    try:
        cat = grid.catchment(x=lon, y=lat, fdir=fdir, xytype="coordinate")
        cat = np.asarray(cat, dtype=bool)[:r, :c]
    except Exception as e:
        print("catchment failed for sink", s.sink_id, e); continue
    free = cat & (catch_labels == 0)                  # larger sinks claimed first
    catch_labels[free] = int(s.sink_id)
    catch_areas[int(s.sink_id)] = float(cat.sum() * CELL_AREA)

top["catchment_m2"] = top.sink_id.map(catch_areas).fillna(0)
top.to_csv(f"{WORK}/sinks.csv", index=False)

meta = dict(driver="GTiff", height=r, width=c, count=1, crs="EPSG:4326",
            transform=transform)
with rasterio.open(f"{WORK}/catchment_labels.tif", "w", dtype="int32", **meta) as dst:
    dst.write(catch_labels, 1)
with rasterio.open(f"{WORK}/depth.tif", "w", dtype="float32", **meta) as dst:
    dst.write(depth_a.astype("float32"), 1)
with rasterio.open(f"{WORK}/acc.tif", "w", dtype="float32", **meta) as dst:
    dst.write(np.asarray(acc, dtype="float32")[:r, :c], 1)
print("saved sinks.csv, catchment_labels.tif, depth.tif, acc.tif")

## Interactive map

In [ ]:
import folium, json
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from PIL import Image

# colour-mapped PNG of ponding depth for the overlay
norm = mcolors.Normalize(vmin=0.0, vmax=max(0.5, float(np.nanpercentile(depth_a, 99.9))))
rgba = cm.Blues(norm(np.nan_to_num(depth_a)))
rgba[..., 3] = np.where(depth_a > MIN_DEPTH, 0.75, 0.0)
Image.fromarray((rgba * 255).astype(np.uint8)).save(f"{WORK}/depth_overlay.png")

lon0, lat1 = transform * (0, 0)
lon1, lat0 = transform * (c, r)
m = folium.Map(location=[(lat0 + lat1) / 2, (lon0 + lon1) / 2], zoom_start=12, tiles="cartodbpositron")
folium.raster_layers.ImageOverlay(f"{WORK}/depth_overlay.png",
                                  bounds=[[lat0, lon0], [lat1, lon1]], opacity=0.8).add_to(m)
features = []
for _, s in top.iterrows():
    folium.CircleMarker([s.lat, s.lon], radius=6, color="crimson", fill=True,
        popup=(f"sink {int(s.sink_id)}<br>volume {s.volume_m3:,.0f} m³"
               f"<br>max depth {s.max_depth_m} m<br>catchment {s.catchment_m2/1e6:.2f} km²")
    ).add_to(m)
    features.append(dict(type="Feature",
        geometry=dict(type="Point", coordinates=[float(s.lon), float(s.lat)]),
        properties=dict(sink_id=int(s.sink_id), volume_m3=float(s.volume_m3),
                        max_depth_m=float(s.max_depth_m), catchment_m2=float(s.catchment_m2))))
with open(f"{WORK}/sinks.geojson", "w") as f:
    json.dump(dict(type="FeatureCollection", features=features), f)
m.save(f"{WORK}/sink_map.html")
m

## Caveats (read before trusting any single point)
- Even FABDEM carries ~1–2 m vertical error; in pancake-flat Patna that misranks shallow sinks.
  Treat this as a **shortlist generator**, then verify top sites on the ground and with
  notebook 03's radar observations.
- Storm sewers and pumped drainage are invisible to a DEM. Sites near major drains may not
  pond in reality.
- Re-run with CartoDEM 10 m from Bhuvan/Bhoonidhi for a sharper pass once registered.